# 02 — Compare models

Runs every registered model on poetry + prose, scores them via the official scorer, and plots a comparison.

## How to add your own model

1. Edit [`src/latinbench/models/template.py`](../src/latinbench/models/template.py) — subclass `Model`, set `name`, implement `predict`.
2. Re-run the cells below.

Results are cached under `predictions/<model.name>/scores.json`. While iterating, either bump `MyModel.name` or pass `force=True` to skip the cache.

In [ ]:
%load_ext autoreload
%autoreload 2

from latinbench import Bench, MODELS
from latinbench.models.template import MyModel
from latinbench.models import UdpipeModel


## Explore registered models

Peek at what's available and which version of each will run before kicking off the bench.

In [ ]:
from latinbench.models.udpipe import list_perseus_models, DEFAULT_MODEL_ID

print('Registered models (MODELS):')
for key in ('udpipe', 'latinpipe', 'qwen3-lmstudio'):
    m = MODELS[key]
    print(f'  {key:14s}  -> {m.name}')
print(f'  {MyModel.name:14s}  (local template — edit src/latinbench/models/template.py)')

print()
print('All available UDPipe versions on LINDAT:')
for mid in list_perseus_models():
    marker = '*' if mid == DEFAULT_MODEL_ID else ' '
    print(f' {marker} {mid}')
print()
print("* = current default. Use UdpipeModel('<id>') to try a different one.")

## Run all three models

In [ ]:

bench = Bench()
df = bench.compare([
    UdpipeModel('latin-perseus-ud-2.15-241121'),
    UdpipeModel('latin-perseus-ud-2.17-251125'),
    MODELS['latinpipe'],
    MODELS['qwen3-lmstudio'],
])
df

## LAS comparison

In [ ]:
bench.plot(df, metric='LAS')

## CLAS comparison

In [ ]:
bench.plot(df, metric='CLAS')

## Force a re-run for one model

While iterating on `MyModel`, either bump `name` (results cached per name) or:

In [ ]:
bench.run(MyModel(), force=True)

## Running just one model

In [ ]:
bench.run(MODELS['latinpipe'])  # cached after the first run

## Try the local LLM (LM Studio)

A third reference model, `qwen3-lmstudio`, runs **Qwen3-0.6B locally via LM Studio** (MLX runtime on Apple Silicon) and parses each sentence with constrained JSON-schema structured output. It's intentionally a weak baseline (0.6B params on a hard structural task) — included as a plumbing example for LLM-based parsing rather than for the leaderboard.

**One-time setup** (skip if already done):

1. Install [LM Studio](https://lmstudio.ai/).
2. In the **Discover** tab, search for `Qwen3 0.6B MLX` and download `lmstudio-community/Qwen3-0.6B-MLX-4bit` (~400 MB).
3. Go to the **Developer** tab → load the model → click **Start Server** (defaults to port 1234).
4. Recommended server settings: max parallel requests → 8, Flash Attention → on, KV cache → q8_0.
5. Sanity check: `curl http://localhost:1234/v1/models` should list the loaded model.

The cell below runs the full bench on the LLM. Expect **many minutes** on a 0.6B model — results cache to `predictions/<slug>/scores.json`, so re-runs are instant. The end-of-run summary reports how many tokens fell back (invalid head, malformed output) separately from LAS/CLAS.

In [ ]:
bench.run(MODELS['qwen3-lmstudio'])  # cached after the first run

# To try a different model loaded in LM Studio (each model id gets its own
# predictions/<slug>/ cache dir; pass the exact id LM Studio reports for
# `GET /v1/models`):
# from latinbench.models.lmstudio_llm import LMStudioModel
# bench.run(LMStudioModel('qwen/qwen3-4b'))
# bench.run(LMStudioModel('qwen/qwen3-8b'))

## Why does qwen3-lmstudio do so poorly? (LLM error analysis)

Run this cell once `qwen3-lmstudio` has finished scoring. It dissects the predictions
to show *how* the LLM fails — not just *that* it fails. Three pieces:

- **Per-token error breakdown.** Of the four possible outcomes (both right / only head right / only label right / both wrong), where do most tokens land?
- **Label distribution: gold vs predicted.** Reveals whether the model is hallucinating a tiny vocabulary of labels (English-style `det`, `nsubj`) rather than picking from the full UD inventory.
- **Per-relation accuracy.** For each gold relation type, how often does the model get the parent right? The label?


In [ ]:
import conllu
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

from latinbench.data import gold_path, PREDICTIONS_DIR

PRED_DIR = PREDICTIONS_DIR / "qwen-qwen3-0.6b"

def load_paired(split):
    g = conllu.parse(Path(gold_path(split)).read_text())
    p = conllu.parse((PRED_DIR / f"{split}_pred.conllu").read_text())
    rows = []
    for gs, ps in zip(g, p):
        sw_g = [t for t in gs if isinstance(t["id"], int)]
        sw_p = [t for t in ps if isinstance(t["id"], int)]
        for gt, pt in zip(sw_g, sw_p):
            rows.append({
                "split": split,
                "tok_id": gt["id"],
                "gold_head": gt["head"],
                "gold_deprel": (gt["deprel"] or "").split(":")[0],
                "pred_head": pt["head"],
                "pred_deprel": (pt["deprel"] or "").split(":")[0],
            })
    return pd.DataFrame(rows)

df = pd.concat([load_paired(s) for s in ("poetry", "prose")], ignore_index=True)
df["head_ok"]  = df["gold_head"]  == df["pred_head"]
df["label_ok"] = df["gold_deprel"] == df["pred_deprel"]

def err(r):
    if r["head_ok"] and r["label_ok"]: return "correct"
    if r["head_ok"]:                   return "head right, label wrong"
    if r["label_ok"]:                  return "label right, head wrong"
    return "both wrong"
df["error_type"] = df.apply(err, axis=1)

# --- 1. Per-token error breakdown ---
err_pct = (df.groupby(["split", "error_type"]).size().unstack(fill_value=0)
           .pipe(lambda x: x.div(x.sum(axis=1), axis=0) * 100).round(1))
print("Per-token error breakdown (%):")
print(err_pct)
print()

# --- 2. Label distribution: gold vs predicted ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, split in zip(axes, ("poetry", "prose")):
    sub = df[df["split"] == split]
    gold_c = Counter(sub["gold_deprel"])
    pred_c = Counter(sub["pred_deprel"])
    labels = [l for l, _ in (gold_c + pred_c).most_common(10)]
    n = len(sub)
    g_vals = [100 * gold_c[l] / n for l in labels]
    p_vals = [100 * pred_c[l] / n for l in labels]
    x = range(len(labels))
    ax.bar([i - 0.2 for i in x], g_vals, width=0.4, label="gold")
    ax.bar([i + 0.2 for i in x], p_vals, width=0.4, label="qwen3-lmstudio")
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("% of tokens")
    ax.set_title(f"{split}: top-10 deprels (gold vs predicted)")
    ax.legend()
plt.tight_layout()
plt.show()

# --- 3. Per-relation accuracy ---
rel = (df.groupby(["split", "gold_deprel"])
         .agg(n=("head_ok", "size"),
              head_acc=("head_ok", "mean"),
              label_acc=("label_ok", "mean"))
         .reset_index())
rel = rel[rel["n"] >= 50].sort_values(["split", "n"], ascending=[True, False])
rel["head_acc"]  = (rel["head_acc"]  * 100).round(1)
rel["label_acc"] = (rel["label_acc"] * 100).round(1)
print("Per-relation accuracy (gold relations with >=50 tokens):")
with pd.option_context("display.max_rows", None, "display.width", 140):
    print(rel.to_string(index=False))

### What the analysis tells us

**1. The model is essentially producing noise on structure.**
- ~80% of tokens have *both* head and label wrong.
- Only ~1-2% of tokens are fully correct.
- The "label right, head wrong" bucket (~6%) is bigger than the "head right, label wrong" bucket — i.e. when it picks a label that happens to match gold, it's mostly a frequency-bias coincidence, not because it understood the dependency.

**2. Label collapse: the model has a tiny working vocabulary.**

Look at the gold vs predicted bars. The model over-predicts a handful of common English-flavored labels and is essentially never producing the rest:
- `det`: predicted ~20% of tokens, gold ~4%. Latin has no articles — every "det" is a hallucination.
- `nsubj`: predicted ~25%, gold ~10%. The model labels almost every nominal as a subject.
- Many real Latin relations (`amod`, `conj`, `case`, `nmod`, `cop`, `acl`, `mark`…) have **0.0% label accuracy** in the per-relation table — the model never picks those labels at all.

**3. Structural prediction is uniformly weak.**

Head accuracy is 5-25% across every relation type — no pocket of competence. Compare LatinPipe, which gets ~75% LAS overall: its per-relation head accuracy is 60-90% across the board.

**4. Why it happens.**

- **0.6B parameters is far below what's needed for syntactic competence on a non-English language.** Modern dependency parsers (UDPipe 2, LatinPipe) are tens-of-millions-of-params transformers *trained directly on Latin UD trees*. A general-purpose chat LLM that small simply has no Latin syntax representation to draw on.
- **Constrained decoding only fixes the symptoms.** The JSON schema forces valid UD labels and integer heads, but it can't put structure into output the model doesn't have. We end up with nicely-formatted nonsense.
- **No global tree constraint.** Each token's head is predicted independently — the model has no way to enforce "exactly one root, no cycles". Our `_is_valid_tree` repair catches the worst cases but right-branches them, which is why the `dep` label appears 22-30% of the time in predictions.

**5. What would help.**

- Bigger model: `qwen3:4b` / `qwen3:8b` would already do meaningfully better.
- Few-shot examples of Latin parses in the prompt (the current prompt has one English-style example).
- Asking the model to score each candidate head + applying Chu-Liu-Edmonds (the `ufal.chu-liu-edmonds` dep is already installed) to extract the maximum-spanning tree — turns N independent decisions into one global decision.

But the easiest experiment is just to swap the model id; everything else stays the same.
